# Sinhala OCR accuracy pilot — Google Colab
Your laptop only needs a browser. This notebook runs **one open-weight OCR model at a time on Google's GPU**, saving results to your Google Drive. It does not train the final model.

**Before opening:** commit/push the code on your current branch. Upload the entire local `reviews/ocr/pilot_20261011` folder (including `images`, which Git ignores) to `My Drive/sinhala-corpus/reviews/ocr/pilot_20261011`.

Choose **Runtime → Change runtime type → GPU**. Start with **Gemma**. Click the triangle beside each code cell from top to bottom. GPU availability varies; this workflow has not yet been tested on a Colab GPU. See `docs/COLAB_OCR_GUIDE.md`.

Use only public corpus material. Never upload hidden benchmark questions. Reference files start empty; model outputs remain unreviewed and cannot be treated as accurate training data.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Choose the model and paths
Keep `BACKEND = "gemma"` for your first run. Start a fresh runtime for a separate DeepSeek run. DeepSeek upstream requires BF16 plus Flash Attention; use an L4/A100-class GPU, not T4. Neither model's Sinhala accuracy has been measured yet.

In [ ]:
from pathlib import Path
import subprocess, sys, json, os, datetime
BACKEND = 'gemma'  # 'deepseek' only with BF16-capable GPU
BRANCH = 'feat/pdf-extraction'  # change if you later merge the notebook elsewhere
REPOSITORY = 'https://github.com/Warusha-Shamalka/sinhala-corpus.git'
ROOT = Path('/content/sinhala-corpus')
DRIVE_ROOT = Path('/content/drive/MyDrive/sinhala-corpus')
PACKET = DRIVE_ROOT / 'reviews/ocr/pilot_20261011'
assert (PACKET / 'manifest.json').is_file(), 'Upload the complete review folder to the Drive path above first'
subprocess.run(['nvidia-smi'], check=True)
if ROOT.exists():
    raise RuntimeError('Repository folder already exists. Reconnect to a fresh runtime, or inspect it before rerunning setup.')
subprocess.run(['git','clone','--branch',BRANCH,'--single-branch',REPOSITORY,str(ROOT)],check=True)
print('Repository commit:', subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip())

## Install an isolated environment
These packages can take several minutes to install. Core pipeline dependencies stay small. Gemma uses the runtime's installed Torch; DeepSeek uses its upstream Torch/Transformers versions. First Gemma installation resolves current compatible packages; the run saves the exact package versions. Use that saved lock file to reproduce the run.

If the GPU check or package installation fails, stop and share the error. Do not run the whole notebook repeatedly or purchase compute before checking the failure.

In [ ]:
ENV = Path('/content/ocr-env')
# Some hosted Python installations lack ensurepip/venv support.
creation = subprocess.run(
    [sys.executable, '-m', 'venv', '--system-site-packages', str(ENV)],
    capture_output=True, text=True,
)
if creation.returncode:
    print('Standard venv creation failed:')
    print(creation.stdout)
    print(creation.stderr)
    print('Trying virtualenv with the same runtime Python...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'virtualenv'], check=True)
    subprocess.run([
        sys.executable, '-m', 'virtualenv', '--python', sys.executable,
        '--system-site-packages', str(ENV),
    ], check=True)
PYTHON = str(ENV / 'bin/python')
subprocess.run([PYTHON, '-m', 'pip', '--version'], check=True)
def pip(*packages):
    subprocess.run([PYTHON,'-m','pip','install',*packages],check=True)
pip('PyYAML==6.0.3','huggingface_hub','Pillow')
if BACKEND == 'gemma':
    pip('--upgrade','transformers','accelerate')
elif BACKEND == 'deepseek':
    subprocess.run([PYTHON,'-c',"import torch; assert torch.cuda.get_device_capability(0)[0] >= 8 and torch.cuda.is_bf16_supported(including_emulation=False), 'DeepSeek needs BF16: use L4/A100, not T4'"],check=True)
    pip('torch==2.6.0','torchvision==0.21.0','--index-url','https://download.pytorch.org/whl/cu118')
    pip('transformers==4.46.3','tokenizers==0.20.3','einops','easydict','addict','PyMuPDF','img2pdf','numpy','ninja','packaging')
    pip('flash-attn==2.7.3','--no-build-isolation')
else:
    raise ValueError('Unknown backend')
subprocess.run([PYTHON,'-B','-m','pipeline.quality.ocr_benchmark','status',str(PACKET)],cwd=ROOT,check=True)

## Hugging Face access and download
For Gemma, visit its Hugging Face model page, sign in, and accept any displayed access conditions. Create a read token in Hugging Face Settings → Access Tokens. Add it as `HF_TOKEN` in Colab's **Secrets** panel (key icon), granting this notebook access. Do not paste the token into a code cell or commit it. A public model may not require a token.

This downloads open weights; transcription uses local GPU weights, not a hosted inference API. Model caches use temporary `/content`; results and the revision manifest use Drive. Reconnecting may require another model download.

In [ ]:
from google.colab import userdata
try:
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = None
MODEL_ID = {'gemma':'google/gemma-4-E2B-it','deepseek':'deepseek-ai/DeepSeek-OCR-2'}[BACKEND]
RUN = DRIVE_ROOT / 'ocr-comparison' / (BACKEND + '_' + datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
RUN.mkdir(parents=True,exist_ok=False)
run_env = os.environ.copy()
if hf_token:
    run_env['HF_TOKEN'] = hf_token
run_env['HF_HOME'] = '/content/hf-cache'
selection_code = r"""
import json, sys
from pathlib import Path
from huggingface_hub import HfApi, snapshot_download
model_id, output = sys.argv[1:]
revision = HfApi().model_info(model_id).sha
path = snapshot_download(model_id, revision=revision)
Path(output).write_text(json.dumps({'model_id':model_id,'revision':revision,'path':path},indent=2)+'\n')
"""
subprocess.run([PYTHON,'-c',selection_code,MODEL_ID,str(RUN/'model_selection.json')],env=run_env,check=True)
selection = json.loads((RUN/'model_selection.json').read_text())
(RUN/'requirements.lock.txt').write_text(subprocess.check_output([PYTHON,'-m','pip','freeze'],text=True))
(RUN/'repository_commit.txt').write_text(subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True))
print('Model revision:',selection['revision'])
print('Results folder:',RUN)

## Transcribe the same ten source images
The runner checks image hashes, never reads reference text, reports each completed page, and saves each prediction immediately to Drive. A new run preserves previous outputs. Start with `MAX_PAGES = 1` if you want a quick GPU compatibility check; later use 10 in a new output directory.

First GPU run is experimental. Out-of-memory, incompatible libraries and truncated output are recorded as failures/incomplete evidence, never as accepted training data.
Gemma now has an exact-token repetition-loop guard and a 120-second soft generation deadline. Stopped output is preserved as truncated and excluded from accuracy scoring; the deadline excludes loading and image preparation and can exceed 120 seconds while a GPU step finishes. DeepSeek upstream infer does not use this guard.


In [ ]:
MAX_PAGES = 10
PREDICTIONS = RUN / 'predictions'
subprocess.run([PYTHON,'-B','-m','pipeline.ocr.neural',
    '--packet',str(PACKET),'--output',str(PREDICTIONS),'--backend',BACKEND,
    '--model-path',selection['path'],'--revision',selection['revision'],
    '--max-pages',str(MAX_PAGES),'--max-new-tokens','8192',
    '--max-generation-seconds','120'],
    cwd=ROOT,env={**run_env,'HF_HUB_OFFLINE':'1','TRANSFORMERS_OFFLINE':'1'},check=True)

## Score only independently verified references
Pending human references produce **pending_references**, not an accuracy estimate. Follow `docs/OCR_ACCURACY_GUIDE.md` to transcribe and independently verify the images before comparing results. The same source document belongs entirely to calibration or holdout.

DeepSeek's upstream API hides generation token counts, so its completion cannot yet be checked automatically. Those outputs remain `completion_unverified` and are excluded from scored results. This is an explicit remaining adapter limitation, not evidence that DeepSeek is inaccurate.

In [ ]:
report = subprocess.check_output([PYTHON,'-B','-m','pipeline.quality.ocr_benchmark',
    'evaluate',str(PACKET),'--predictions',str(PREDICTIONS)],cwd=ROOT,text=True)
(RUN/'accuracy_report.json').write_text(report,encoding='utf-8')
print(report)
print('Saved on Google Drive:',RUN)

## Finish
Inspect the saved page JSON files alongside source images. Do not copy unreviewed output into training datasets. Share the run folder and errors with your team, keeping tokens out of saved outputs. Select **Runtime → Disconnect and delete runtime** when finished; files already saved in Drive remain available.

For a second backend, use a new runtime and change BACKEND. After human references are verified, run the evaluator with two `--predictions` directories to compare identical pages. Do not tune prompts on the holdout pages.